In [3]:
import pyspark
from pyspark.sql import SparkSession
import os
import sys

In [4]:
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ["JAVA_HOME"] = "C:/tools/jdk-17.0.18+8"
os.environ["HADOOP_HOME"] = "C:/tools/hadoop"
os.environ['PATH'] = os.environ.get('PATH', '') + ';' + 'C:\\tools\\hadoop\\bin'
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

In [5]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

In [6]:
spark.version


'4.1.1'

In [17]:
df_yellow = spark.read.parquet(r"C:\Users\USER\Desktop\Python\Projects\data-engineering-zoomcamp\module-6-batch\data\raw\yellow\2025\11\yellow_tripdata_2025-11.parquet")

In [ ]:
output_path = r"C:\Users\USER\Desktop\Python\Projects\data-engineering-zoomcamp\module-6-batch\data\pq\yellow\2025\11\ "

In [21]:
df_yellow.repartition(4).write.parquet(r'C:\Users\USER\Desktop\Python\Projects\data-engineering-zoomcamp\module-6-batch\data\pq\yellow\2025\11', mode="overwrite")

In [24]:
from pyspark.sql import functions as F

In [28]:
df_yellow.createOrReplaceTempView("yellow_table")

In [49]:
df_yellow.select('DOLocationID').show()

+------------+
|DOLocationID|
+------------+
|         186|
|         237|
|         238|
|         261|
|          37|
|         100|
|         170|
|         144|
|         161|
|         162|
|          88|
|         148|
|         236|
|         255|
|          43|
|         262|
|          24|
|         147|
|         137|
|         237|
+------------+
only showing top 20 rows


In [44]:
spark.sql(""" 
SELECT tpep_dropoff_datetime - tpep_pickup_datetime AS length
FROM yellow_table
ORDER BY length DESC
LIMIT 5;
""").show()

+--------------------+
|              length|
+--------------------+
|INTERVAL '3 18:38...|
|INTERVAL '3 04:56...|
|INTERVAL '3 04:12...|
|INTERVAL '2 21:17...|
|INTERVAL '2 19:04...|
+--------------------+



In [59]:
taxi_zone = spark.read\
    .option("header","true")\
    .csv(r'C:\Users\USER\Desktop\Python\Projects\data-engineering-zoomcamp\module-6-batch\data\taxi_zone_lookup.csv')

In [60]:
taxi_zone.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [61]:
taxi_zone.createOrReplaceTempView("taxi_zones")

In [69]:
spark.sql(""" 
SELECT Zone, COUNT(*) as num
FROM yellow_table AS y
JOIN taxi_zones AS t
ON t.LocationID = y.PULocationID
GROUP BY Zone
ORDER BY num
""").show(truncate=False)

+---------------------------------------------+---+
|Zone                                         |num|
+---------------------------------------------+---+
|Governor's Island/Ellis Island/Liberty Island|1  |
|Eltingville/Annadale/Prince's Bay            |1  |
|Arden Heights                                |1  |
|Port Richmond                                |3  |
|Rikers Island                                |4  |
|Rossville/Woodrow                            |4  |
|Green-Wood Cemetery                          |4  |
|Great Kills                                  |4  |
|Jamaica Bay                                  |5  |
|Westerleigh                                  |12 |
|Crotona Park                                 |14 |
|Oakwood                                      |14 |
|New Dorp/Midland Beach                       |14 |
|West Brighton                                |14 |
|Willets Point                                |15 |
|Breezy Point/Fort Tilden/Riis Beach          |16 |
|Saint Georg